# Construyendo el Backend para una Aplicación Full-Stack con LangGraph

## Guía Paso a Paso: Del Agente en Notebook al Backend con FastAPI

En el notebook `028-1st-top-use-case-langgraph-2026.ipynb`, construimos un **agente de atención al cliente con LangGraph** para BigCompany Inc. que se ejecuta dentro de un notebook Jupyter.

Eso está bien para aprender y probar, pero en el mundo real, un agente de IA necesita ser **accesible desde la web**. Un cliente debería poder chatear con el agente a través de un sitio web, no a través de un notebook Jupyter.

Para conseguirlo, necesitamos envolver nuestro agente de LangGraph dentro de un **servidor backend** — igual que la aplicación To Do en la carpeta `050-todoapp-LC-backend` usa FastAPI para exponer sus funcionalidades como una API.

**En este notebook, crearemos el backend paso a paso.**

## ¿Qué es un Backend? (Repaso rápido)

Piénsalo de esta manera:

- **El agente de LangGraph** es el "cerebro" — sabe cómo responder preguntas de clientes, consultar pedidos y procesar devoluciones.
- **El backend (FastAPI)** es la "recepción" — recibe mensajes del sitio web, se los pasa al cerebro y devuelve la respuesta.
- **El frontend** (que construiremos después) es la "ventana de chat" en el sitio web que el cliente ve.

```
┌──────────────┐     Petición HTTP      ┌──────────────────┐     invoke()     ┌──────────────────┐
│              │  ─────────────────►    │                  │  ─────────────►  │                  │
│   Frontend   │   POST /chat           │  Backend FastAPI │                  │  Agente LangGraph│
│  (Sitio web) │   {message, thread}    │  (Recepción)     │                  │  (Cerebro)       │
│              │  ◄─────────────────    │                  │  ◄─────────────  │                  │
└──────────────┘     Respuesta HTTP     └──────────────────┘     respuesta    └──────────────────┘
```

El backend es simplemente un **puente** entre el frontend y el agente.

## ¿Cómo se compara con el Backend básico de la aplicación To Do que construimos en el Bootcamp #1?

En la aplicación To Do (`050-todoapp-LC-backend`), el backend:
- Usa **FastAPI** como framework web.
- Tiene **endpoints de API** (como `POST /todos`, `GET /todos`) para crear y leer tareas.
- Usa **LangChain** para añadir funcionalidades de IA (resumir texto, escribir poemas).
- Se conecta a una **base de datos PostgreSQL** con SQLAlchemy.

Nuestro nuevo backend será **más sencillo**:
- Usa **FastAPI** como framework web (igual).
- Tiene **un endpoint principal**: `POST /chat` para enviar mensajes al agente.
- Usa **LangGraph** en lugar de LangChain (el agente del notebook 028).
- **No necesita base de datos** — la memoria integrada de LangGraph gestiona el historial de conversaciones.

| Característica | Backend App To Do (050) | Nuestro Backend LangGraph |
|----------------|------------------------|---------------------------|
| Framework web | FastAPI | FastAPI |
| Framework IA | LangChain (cadena simple) | LangGraph (agente con herramientas) |
| Base de datos | PostgreSQL + SQLAlchemy | Ninguna (estado en memoria) |
| Endpoint principal | Endpoints CRUD (`/todos`) | Endpoint de chat (`/chat`) |
| Middleware CORS | Sí | Sí |

## Estructura del Proyecto

Estos son los archivos que crearemos:

```
backend/
├── .env                  # Claves secretas (clave API de OpenAI)
├── requirements.txt      # Paquetes de Python que necesitamos
├── main.py               # Aplicación FastAPI (la "recepción")
├── agent.py              # Agente LangGraph (el "cerebro")
├── schemas.py            # Formas de datos para peticiones y respuestas
└── routers/
    └── chat.py           # El endpoint de la API /chat
```

Compáralo con la aplicación To Do (`050-todoapp-LC-backend`):

```
050-todoapp-LC-backend/
├── .env                  # Claves secretas
├── pyproject.toml        # Paquetes de Python
├── main.py               # Aplicación FastAPI       ← mismo rol
├── config.py             # Configuración
├── database.py           # Conexión a base de datos  ← no lo necesitamos
├── models.py             # Modelos de base de datos  ← no lo necesitamos
├── schemas.py            # Formas de datos           ← mismo rol
├── crud.py               # Operaciones de base de datos ← no lo necesitamos
└── routers/
    └── todos.py          # Endpoints de API          ← mismo rol (el nuestro: chat.py)
```

¿Ves el patrón? Nuestro backend es **más sencillo** porque no necesitamos base de datos. El agente de LangGraph gestiona su propio estado.

¡Vamos a construirlo archivo por archivo!

## Paso 1: Crear la Carpeta del Proyecto

Primero, vamos a crear la carpeta `backend/` y la subcarpeta `routers/` dentro de ella.

In [ ]:
import os

# Crear la carpeta backend y la subcarpeta routers
os.makedirs("backend/routers", exist_ok=True)

print("¡Carpetas creadas!")
print("  backend/")
print("  backend/routers/")

## Paso 2: Crear el Archivo `.env` (Claves Secretas)

El archivo `.env` almacena nuestra **clave API de OpenAI**. Esta clave es necesaria para que el LLM (el cerebro del agente) funcione.

**Importante**: ¡Nunca compartas tu clave API ni la subas a Git!

En la aplicación To Do, el archivo `.env` tenía credenciales de base de datos Y una clave de OpenAI. El nuestro solo necesita la clave de OpenAI porque no usamos base de datos.

In [ ]:
%%writefile backend/.env
OPENAI_API_KEY=tu-clave-api-de-openai-aqui

**Después de ejecutar esta celda**, abre el archivo `backend/.env` y reemplaza `tu-clave-api-de-openai-aqui` con tu clave API real de OpenAI.

## Paso 3: Crear `requirements.txt` (Dependencias)

Este archivo lista todos los paquetes de Python que nuestro backend necesita.

| Paquete | Para qué sirve |
|---------|----------------|
| `fastapi` | El framework web (gestiona peticiones HTTP) |
| `uvicorn[standard]` | El servidor que ejecuta nuestra aplicación FastAPI |
| `langgraph` | El framework para el flujo de trabajo de nuestro agente IA |
| `langchain-openai` | Se conecta a los modelos GPT de OpenAI |
| `langchain-core` | Utilidades básicas (tipos de mensajes, herramientas) |
| `python-dotenv` | Carga el archivo `.env` |
| `pydantic` | Validación de datos para peticiones/respuestas |

En la aplicación To Do, estos estaban listados en `pyproject.toml` (gestionado por Poetry). Para simplificar, usaremos un simple `requirements.txt` (gestionado por pip).

In [ ]:
%%writefile backend/requirements.txt
fastapi>=0.115.0
uvicorn[standard]>=0.34.0
langgraph>=0.2.0
langchain-openai>=0.3.0
langchain-core>=0.3.0
python-dotenv>=1.0.0
pydantic>=2.0.0

Para instalar estos paquetes, ejecuta este comando en tu terminal:

```bash
pip install -r backend/requirements.txt
```

## Paso 4: Crear `agent.py` (El Agente LangGraph)

Este es el **corazón de nuestro backend**. Contiene el mismo agente de LangGraph que construimos en el notebook 028, pero organizado en un único archivo Python que puede ser importado por la API.

Esto es lo que hace este archivo:

1. **Define la base de datos simulada** (pedidos, política de devoluciones) — igual que en el notebook 028.
2. **Define las herramientas** (check_order_status, lookup_return_policy, process_return_request) — igual que en el notebook 028.
3. **Configura el LLM** con las herramientas vinculadas — igual que en el notebook 028.
4. **Construye el grafo de LangGraph** (nodo asistente, nodo herramientas, aristas condicionales) — igual que en el notebook 028.
5. **Exporta el objeto `graph`** para que la API pueda usarlo.

La única diferencia con el notebook 028: todo el código está en **un solo archivo** en lugar de repartido en múltiples celdas del notebook.

In [ ]:
%%writefile backend/agent.py
# ============================================================
# agent.py — El Agente de Atención al Cliente con LangGraph
#
# Este archivo contiene el MISMO agente del notebook 028,
# empaquetado como un módulo que el backend de FastAPI puede importar.
# ============================================================

import os
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import InMemorySaver

# Cargar variables de entorno (la clave API de OpenAI)
load_dotenv()


# ============================================================
# 1. LA "BASE DE DATOS" (simulada con diccionarios)
# ============================================================
# En una aplicación real, estos datos vendrían de una base de datos real.
# Para esta demostración, usamos simples diccionarios de Python.

ORDERS_DB = {
    "ORD-1001": {
        "customer": "Alice Johnson",
        "product": "Laptop Pro 15",
        "price": 1299.99,
        "status": "delivered",
        "delivery_date": "2026-02-05",
    },
    "ORD-1002": {
        "customer": "Bob Smith",
        "product": "Wireless Headphones X3",
        "price": 199.99,
        "status": "shipped",
        "estimated_delivery": "2026-02-14",
    },
    "ORD-1003": {
        "customer": "Carol Davis",
        "product": "Smartphone Ultra 12",
        "price": 899.99,
        "status": "processing",
        "estimated_delivery": "2026-02-18",
    },
}

RETURN_POLICY = {
    "standard_return_window": "30 days from delivery",
    "refund_processing_time": "5-7 business days",
    "conditions": [
        "Item must be in original packaging",
        "Item must not show signs of damage caused by the customer",
        "Proof of purchase (order ID) is required",
    ],
    "non_returnable": ["Earbuds (hygiene reasons)", "Software licenses"],
}


# ============================================================
# 2. LAS HERRAMIENTAS (funciones que el agente puede llamar)
# ============================================================
# Estas son las mismas 3 herramientas del notebook 028.
# El decorador @tool las hace utilizables por el LLM.

@tool
def check_order_status(order_id: str) -> str:
    """Check the current status of a customer order.

    Use this tool when a customer asks about their order status,
    delivery date, or order details.

    Args:
        order_id: The order ID (e.g., 'ORD-1001').
    """
    order = ORDERS_DB.get(order_id)

    if not order:
        return f"Order '{order_id}' not found. Please check the order ID and try again."

    info = f"Order {order_id}:\n"
    info += f"  Customer: {order['customer']}\n"
    info += f"  Product: {order['product']}\n"
    info += f"  Price: ${order['price']}\n"
    info += f"  Status: {order['status']}\n"

    if order["status"] == "delivered":
        info += f"  Delivered on: {order['delivery_date']}"
    else:
        info += f"  Estimated delivery: {order['estimated_delivery']}"

    return info


@tool
def lookup_return_policy() -> str:
    """Look up BigCompany Inc.'s return and refund policy.

    Use this tool when a customer asks about returns, refunds,
    or the return policy.
    """
    policy = "BigCompany Inc. Return Policy:\n"
    policy += f"  Return window: {RETURN_POLICY['standard_return_window']}\n"
    policy += f"  Refund processing: {RETURN_POLICY['refund_processing_time']}\n"
    policy += "  Conditions:\n"
    for condition in RETURN_POLICY["conditions"]:
        policy += f"    - {condition}\n"
    policy += "  Non-returnable items:\n"
    for item in RETURN_POLICY["non_returnable"]:
        policy += f"    - {item}\n"
    return policy


@tool
def process_return_request(order_id: str, reason: str) -> str:
    """Process a return request for a customer order.

    Use this tool when a customer confirms they want to return a product.
    Only use this AFTER checking the order status and confirming
    the customer wants to proceed.

    Args:
        order_id: The order ID to process the return for.
        reason: The customer's reason for returning the product.
    """
    order = ORDERS_DB.get(order_id)

    if not order:
        return f"Cannot process return: Order '{order_id}' not found."

    if order["status"] != "delivered":
        return (
            f"Cannot process return: Order {order_id} has status '{order['status']}'. "
            f"Returns can only be processed for delivered orders."
        )

    return (
        f"Return request APPROVED for order {order_id} ({order['product']}).\n"
        f"  Reason: {reason}\n"
        f"  Refund amount: ${order['price']}\n"
        f"  Refund will be processed within {RETURN_POLICY['refund_processing_time']}.\n"
        f"  A return shipping label has been sent to the customer's email."
    )


# Recopilar todas las herramientas en una lista
tools = [check_order_status, lookup_return_policy, process_return_request]


# ============================================================
# 3. EL LLM (el cerebro del agente)
# ============================================================

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_with_tools = llm.bind_tools(tools)

SYSTEM_PROMPT = """You are a friendly and helpful customer service agent for BigCompany Inc., \
an online electronics store.

Your responsibilities:
- Help customers check their order status.
- Answer questions about our return and refund policy.
- Process return requests when customers want to return a product.
- Be polite, empathetic, and professional at all times.

Important guidelines:
- Always greet the customer warmly.
- When a customer asks about an order, use the check_order_status tool.
- When a customer asks about returns or refunds, use the lookup_return_policy tool.
- Before processing a return, always confirm with the customer first.
- If you cannot help with something, let the customer know you will \
connect them with a human agent.
- Keep responses concise but helpful.
"""


# ============================================================
# 4. EL GRAFO (el flujo de trabajo del agente)
# ============================================================

# Nodo 1: El Asistente (llama al LLM)
def assistant(state: MessagesState):
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


# Nodo 2: El Ejecutor de Herramientas
tool_node = ToolNode(tools)

# Construir el grafo
builder = StateGraph(MessagesState)
builder.add_node("assistant", assistant)
builder.add_node("tools", tool_node)
builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)
builder.add_edge("tools", "assistant")

# Compilar con memoria (para que el agente recuerde las conversaciones)
memory = InMemorySaver()
graph = builder.compile(checkpointer=memory)

print("¡Agente LangGraph de BigCompany Inc. cargado correctamente!")

### ¿Qué acaba de pasar?

Hemos creado `backend/agent.py` — un único archivo que contiene nuestro agente de LangGraph completo. Es el mismo agente del notebook 028, solo que empaquetado como un módulo de Python.

Lo clave es la **última línea**: `graph = builder.compile(checkpointer=memory)`. Esto crea el objeto `graph` que nuestro endpoint de la API importará y usará.

```python
# Más adelante, en nuestro endpoint de la API, haremos:
from agent import graph

result = graph.invoke({"messages": [...]}, config)
```

## Paso 5: Crear `schemas.py` (Formas de Petición/Respuesta)

Los **esquemas** definen la forma de los datos que entran y salen de nuestra API. Son como un **contrato**: le dicen al frontend exactamente qué enviar y qué esperar de vuelta.

En la aplicación To Do, los esquemas definían `ToDoRequest` (nombre + completado) y `ToDoResponse` (nombre + completado + id).

Para nuestra aplicación de chat, necesitamos:

| Esquema | Campos | Se usa para |
|---------|--------|-------------|
| `ChatRequest` | `message` + `thread_id` | Lo que el frontend **envía** a la API |
| `ChatResponse` | `response` + `thread_id` | Lo que la API **devuelve** al frontend |

El `thread_id` es importante — le dice al agente **a qué conversación** pertenece este mensaje. Mismo thread_id = misma conversación (el agente recuerda los mensajes anteriores).

In [ ]:
%%writefile backend/schemas.py
# ============================================================
# schemas.py — Formas de datos para la API de Chat
#
# Los modelos Pydantic definen qué datos entran y salen de la API.
# Piensa en ellos como un "contrato" entre el frontend y el backend.
# ============================================================

from pydantic import BaseModel


class ChatRequest(BaseModel):
    """Lo que el frontend envía al backend."""
    message: str       # El mensaje del cliente (ej., "¿Cuál es el estado de mi pedido?")
    thread_id: str     # ID de conversación (mismo ID = misma conversación)


class ChatResponse(BaseModel):
    """Lo que el backend devuelve al frontend."""
    response: str      # La respuesta del agente
    thread_id: str     # Mismo ID de conversación (para que el frontend sepa a qué conversación pertenece)

### Comparación con los esquemas de la aplicación To Do

Aplicación To Do (`050-todoapp-LC-backend/schemas.py`):
```python
class ToDoRequest(BaseModel):
    name: str
    completed: bool

class ToDoResponse(BaseModel):
    name: str
    completed: bool
    id: int
```

Nuestra aplicación de chat (`backend/schemas.py`):
```python
class ChatRequest(BaseModel):
    message: str
    thread_id: str

class ChatResponse(BaseModel):
    response: str
    thread_id: str
```

Mismo patrón, campos diferentes. ¡Sencillo!

## Paso 6: Crear `routers/chat.py` (El Endpoint de la API de Chat)

Aquí es donde ocurre la magia. Este archivo crea el endpoint **`POST /chat`** — la puerta a través de la cual el frontend habla con el agente.

En la aplicación To Do, `routers/todos.py` tenía múltiples endpoints (POST, GET, PUT, DELETE para tareas). Nuestra aplicación de chat solo necesita **un endpoint**: `POST /chat`.

Esto es lo que hace el endpoint:

1. **Recibe** el mensaje del cliente y el thread_id desde el frontend.
2. **Pasa** el mensaje al agente de LangGraph.
3. **Espera** a que el agente piense, use herramientas y genere una respuesta.
4. **Devuelve** la respuesta del agente al frontend.

In [ ]:
%%writefile backend/routers/chat.py
# ============================================================
# routers/chat.py — El Endpoint de la API de Chat
#
# Esta es la "puerta" entre el frontend y el agente de LangGraph.
# Recibe mensajes del frontend, se los pasa al agente,
# y devuelve la respuesta del agente.
# ============================================================

from fastapi import APIRouter
from langchain_core.messages import HumanMessage

from schemas import ChatRequest, ChatResponse
from agent import graph

router = APIRouter()


@router.post("/chat", response_model=ChatResponse)
async def chat(request: ChatRequest):
    """
    Envía un mensaje al agente de atención al cliente de BigCompany Inc.

    - El `message` es lo que el cliente ha escrito.
    - El `thread_id` identifica la conversación.
      Mismo thread_id = misma conversación (el agente recuerda el contexto).
    """

    # La configuración le dice a LangGraph qué hilo de conversación usar.
    # Así es como el agente "recuerda" mensajes anteriores.
    config = {"configurable": {"thread_id": request.thread_id}}

    # Enviar el mensaje del cliente al agente de LangGraph.
    # El agente:
    #   1. Lee el mensaje
    #   2. Decide si necesita usar una herramienta
    #   3. Si sí: llama a la herramienta, obtiene resultados, piensa de nuevo
    #   4. Genera una respuesta final
    result = graph.invoke(
        {"messages": [HumanMessage(content=request.message)]},
        config,
    )

    # El resultado contiene TODOS los mensajes de la conversación.
    # El último mensaje es la respuesta final del agente.
    agent_response = result["messages"][-1].content

    return ChatResponse(
        response=agent_response,
        thread_id=request.thread_id,
    )

### Comparación con el router de la aplicación To Do

Aplicación To Do (`050-todoapp-LC-backend/routers/todos.py`) — tenía muchos endpoints:
```python
@router.post("")           # Crear una tarea
@router.get("")            # Listar todas las tareas
@router.get("/{id}")       # Obtener una tarea
@router.put("/{id}")       # Actualizar una tarea
@router.delete("/{id}")    # Eliminar una tarea
@router.post("/summarize-text")  # LangChain: resumir
@router.post("/write-poem/{id}") # LangChain: escribir poema
```

Nuestra aplicación de chat (`backend/routers/chat.py`) — solo uno:
```python
@router.post("/chat")      # Chatear con el agente
```

Un endpoint es todo lo que necesitamos. El agente de LangGraph gestiona toda la complejidad internamente (decidir qué herramienta usar, gestionar el estado de la conversación, etc.).

## Paso 7: Crear `main.py` (La Aplicación FastAPI)

Este es el **punto de entrada** de nuestro backend. Crea la aplicación FastAPI, añade el middleware CORS e incluye el router de chat.

**¿Qué es CORS?** Cuando un frontend (ejecutándose en `localhost:3000`) intenta llamar a un backend (ejecutándose en `localhost:8000`), el navegador lo bloquea por razones de seguridad. El middleware CORS le dice al navegador: *"Está bien, confío en este frontend."*

El `main.py` de la aplicación To Do tenía la misma configuración de CORS. Estamos usando exactamente el mismo patrón.

In [ ]:
%%writefile backend/main.py
# ============================================================
# main.py — La Aplicación FastAPI
#
# Este es el punto de entrada del backend.
# Crea la aplicación FastAPI, añade CORS e incluye el router.
# ============================================================

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

from routers import chat

# Crear la aplicación FastAPI
app = FastAPI(
    title="BigCompany Inc. Customer Service API",
    description="A LangGraph-powered customer service agent backend.",
)

# Middleware CORS — permite que el frontend llame a nuestra API.
# Sin esto, el navegador bloquearía las peticiones del frontend.
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:3000"],  # URL del frontend
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Incluir el router de chat (esto añade el endpoint /chat)
app.include_router(chat.router)


# Un endpoint sencillo de verificación de salud
@app.get("/")
def read_root():
    return {"message": "BigCompany Inc. Customer Service API is running!"}

### Comparación con el `main.py` de la aplicación To Do

La estructura es casi idéntica:

| Parte | Aplicación To Do (050) | Nuestra App de Chat |
|-------|------------------------|---------------------|
| Crear app | `app = FastAPI()` | `app = FastAPI(title=..., description=...)` |
| CORS | `allow_origins=["http://localhost:3000"]` | Igual |
| Incluir router | `app.include_router(todos.router)` | `app.include_router(chat.router)` |
| Endpoint raíz | `return "Hello World"` | `return {"message": "...API is running!"}` |

La aplicación To Do también tenía configuración mediante `config.py` y una conexión a base de datos. No necesitamos eso, así que nuestro `main.py` es más sencillo.

## Paso 8: Crear el archivo `routers/__init__.py`

Este pequeño archivo le dice a Python que la carpeta `routers/` es un paquete que se puede importar. Sin él, `from routers import chat` no funcionaría.

In [ ]:
%%writefile backend/routers/__init__.py
# Este archivo convierte la carpeta routers/ en un paquete de Python.

#### Explicación Sencilla

Piensa en el archivo `__init__.py` como un **cartel en la puerta** de una habitación.

Tienes una carpeta llamada `routers/` con código útil dentro. Python no la reconocerá como un "paquete" propiamente dicho (un grupo de archivos de código relacionados) a menos que pongas un archivo `__init__.py` dentro — aunque ese archivo esté completamente vacío.

**Sin él:** Python dice *"No sé qué es `routers`"* cuando intentas hacer `from routers import chat`.

**Con él:** Python dice *"Ah, `routers` es un paquete, déjame mirar dentro."*

En este caso, el archivo solo tiene un comentario y nada más — su mera **existencia** es lo que importa, no su contenido.

**Es una convención de Python que, aunque no es estrictamente obligatoria en Python 3 moderno, sigue considerándose una buena práctica, especialmente en estructuras de proyecto como los backends de FastAPI**.

## ¡Todos los Archivos Creados! Vamos a Revisar

Verifiquemos que todos nuestros archivos del backend se han creado correctamente.

In [ ]:
import os

print("Estructura del proyecto backend:")
print("=" * 40)
for root, dirs, files in os.walk("backend"):
    # Omitir carpetas ocultas como __pycache__
    dirs[:] = [d for d in dirs if not d.startswith(".") and d != "__pycache__"]
    level = root.replace("backend", "").count(os.sep)
    indent = "  " * level
    folder_name = os.path.basename(root)
    print(f"{indent}{folder_name}/")
    sub_indent = "  " * (level + 1)
    for file in sorted(files):
        if not file.startswith(".") or file == ".env":
            filepath = os.path.join(root, file)
            size = os.path.getsize(filepath)
            print(f"{sub_indent}{file} ({size} bytes)")

## Paso 9: Ejecutar el Backend

¡Ahora vamos a iniciar el servidor! Usaremos **uvicorn**, que es el mismo servidor que usa la aplicación To Do.

### Opción A: Ejecutar desde la terminal

Abre una terminal, navega a la carpeta `backend/` y ejecuta:

```bash
cd backend
uvicorn main:app --reload
```

- `main:app` significa: en el archivo `main.py`, usa el objeto llamado `app`.
- `--reload` significa: reiniciar automáticamente cuando cambies el código (útil durante el desarrollo).

El servidor se iniciará en **http://localhost:8000**.

### Opción B: Ejecutar desde este notebook

También puedes iniciar el servidor desde este notebook (se ejecutará en segundo plano). **Nota**: ¡Asegúrate de haber añadido tu clave API de OpenAI en `backend/.env` primero!

In [ ]:
# Descomenta y ejecuta esta celda para iniciar el servidor desde el notebook.
# Pulsa el botón de parar (o reinicia el kernel) para detenerlo.

# import subprocess
# subprocess.Popen(["uvicorn", "main:app", "--reload"], cwd="backend")

## Paso 10: Probar el Backend

Una vez que el servidor esté funcionando, ¡podemos probarlo! Hay 3 formas de probar:

### Forma 1: Swagger UI (La más fácil)

FastAPI crea automáticamente una página de documentación interactiva muy bonita. Abre tu navegador y ve a:

**http://localhost:8000/docs**

Verás el endpoint `/chat`. Haz clic en él, haz clic en "Try it out" y envía un mensaje de prueba:

```json
{
  "message": "¡Hola! ¿Cuál es el estado del pedido ORD-1002?",
  "thread_id": "test-session-1"
}
```

### Forma 2: Probar desde este notebook con Python

También podemos probar enviando peticiones HTTP desde este notebook (¡asegúrate de que el servidor esté funcionando primero!).

In [ ]:
import requests

API_URL = "http://localhost:8000"

# Prueba 1: Verificación de salud
print("Prueba 1: Verificación de salud")
print("=" * 40)
response = requests.get(f"{API_URL}/")
print(f"Estado: {response.status_code}")
print(f"Respuesta: {response.json()}")
print()

In [ ]:
# Prueba 2: Saludo sencillo
print("Prueba 2: Saludo sencillo")
print("=" * 40)
response = requests.post(
    f"{API_URL}/chat",
    json={
        "message": "¡Hola! Necesito ayuda.",
        "thread_id": "test-session-1",
    },
)
print(f"Estado: {response.status_code}")
data = response.json()
print(f"Agente: {data['response']}")
print(f"Hilo: {data['thread_id']}")
print()

In [ ]:
# Prueba 3: Consultar estado del pedido (el agente usará la herramienta check_order_status)
print("Prueba 3: Consultar estado del pedido")
print("=" * 40)
response = requests.post(
    f"{API_URL}/chat",
    json={
        "message": "¿Cuál es el estado del pedido ORD-1002?",
        "thread_id": "test-session-1",
    },
)
data = response.json()
print(f"Agente: {data['response']}")
print()

In [ ]:
# Prueba 4: Pregunta de seguimiento (el agente debería recordar ORD-1002 del mismo hilo)
print("Prueba 4: Seguimiento (prueba la memoria de conversación)")
print("=" * 40)
response = requests.post(
    f"{API_URL}/chat",
    json={
        "message": "¿Cuándo llegará?",
        "thread_id": "test-session-1",  # ¡Mismo thread_id!
    },
)
data = response.json()
print(f"Agente: {data['response']}")
print()

In [ ]:
# Prueba 5: Pregunta sobre política de devoluciones
print("Prueba 5: Pregunta sobre política de devoluciones")
print("=" * 40)
response = requests.post(
    f"{API_URL}/chat",
    json={
        "message": "¿Cuál es vuestra política de devoluciones?",
        "thread_id": "test-session-2",  # Cliente diferente, hilo diferente
    },
)
data = response.json()
print(f"Agente: {data['response']}")
print()

### Forma 3: Probar con curl (desde la terminal)

También puedes probar desde la terminal usando `curl`:

```bash
curl -X POST http://localhost:8000/chat \
  -H "Content-Type: application/json" \
  -d '{"message": "¿Cuál es el estado del pedido ORD-1001?", "thread_id": "test-1"}'
```

## Resumen: Lo Que Hemos Construido

Hemos creado un **backend con FastAPI** que envuelve el agente de atención al cliente de LangGraph del notebook 028. Aquí tienes un resumen de cada archivo y lo que hace:

| Archivo | Rol | Líneas de código |
|---------|-----|------------------|
| `.env` | Almacena la clave API de OpenAI | 1 |
| `requirements.txt` | Lista los paquetes de Python necesarios | 7 |
| `agent.py` | El agente LangGraph (herramientas, LLM, grafo) — el cerebro | ~140 |
| `schemas.py` | Modelos Pydantic para las formas de petición/respuesta | ~12 |
| `main.py` | Aplicación FastAPI con middleware CORS | ~20 |
| `routers/chat.py` | El endpoint POST /chat de la API | ~25 |
| `routers/__init__.py` | Convierte `routers/` en un paquete de Python | 1 |

### El Flujo (Cómo Se Conecta Todo)

```
El frontend envía POST /chat
  con {message: "Consultar pedido ORD-1002", thread_id: "session-1"}
        │
        ▼
main.py recibe la petición
  └── La enruta a routers/chat.py
        │
        ▼
routers/chat.py
  └── Llama a graph.invoke() de agent.py
        │
        ▼
agent.py (el agente de LangGraph)
  ├── Nodo asistente: el LLM lee el mensaje
  ├── El LLM decide llamar a la herramienta check_order_status
  ├── Nodo herramientas: ejecuta la herramienta, obtiene datos del pedido
  ├── Nodo asistente: el LLM lee los resultados de la herramienta
  └── El LLM genera la respuesta final
        │
        ▼
routers/chat.py
  └── Devuelve {response: "Tu pedido está enviado...", thread_id: "session-1"}
        │
        ▼
El frontend muestra la respuesta del agente
```

## Conclusiones Clave

1. **El backend es solo un puente.** Recibe peticiones HTTP del frontend, se las pasa al agente de LangGraph y devuelve la respuesta. Eso es todo.

2. **El agente es el mismo del notebook 028.** No hemos cambiado el agente en absoluto — simplemente lo hemos movido a su propio archivo (`agent.py`) para que la API pueda importarlo.

3. **FastAPI lo hace fácil.** Crear un endpoint de API con FastAPI es simplemente una función con un decorador (`@router.post("/chat")`). Pydantic gestiona la validación de datos automáticamente.

4. **CORS es esencial.** Sin él, el navegador bloquearía al frontend de llamar a nuestra API. Copia la misma configuración de CORS de la aplicación To Do.

5. **Los thread IDs permiten la memoria de conversación.** Al pasar un `thread_id` con cada mensaje, el agente sabe a qué conversación pertenece el mensaje y puede recordar el contexto.

## ¿Qué Viene Después?

Ahora que el backend está funcionando, el siguiente paso es construir el **frontend** — una interfaz de chat donde los clientes puedan escribir mensajes y ver las respuestas del agente.

El frontend será similar al frontend de la aplicación To Do en `051-todoapp-LC-frontend`, pero en lugar de una lista de tareas, mostrará una **ventana de chat** que envía mensajes a nuestro endpoint `POST /chat`.